# Pluggable PTC tools

Skein selects one model-visible ADK tool while keeping the host effect broker unchanged.

| Implementation | Tool | Python lifetime | Durable representation |
| --- | --- | --- | --- |
| `skein_notebook` | `execute_code` | Persistent run-scoped heap | Write-ahead ledger events plus a deterministic notebook |
| `prime_repl` | `execute_code` | Persistent trusted native process | JSONL cell evidence plus bounded snapshots |

The four direct tools remain the default control arm. Neither PTC tool owns completion.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PtcChoice:
    implementation: str
    tool_name: str
    state_scope: str


CHOICES = {
    "skein_notebook": PtcChoice("skein_notebook", "execute_code", "run"),
    "prime_repl": PtcChoice("prime_repl", "execute_code", "run"),
}


def select_ptc(name):
    return CHOICES[name]


assert select_ptc("skein_notebook").tool_name == "execute_code"
assert select_ptc("prime_repl").tool_name == "execute_code"
print(CHOICES)

## Explicit effect boundaries

Skein notebook calls the broker for file and shell effects. Prime has trusted native access, so its cell is recorded as `native_untracked` rather than falsely represented as brokered calls.


In [ ]:
class Broker:
    def __init__(self):
        self.receipts = []

    def call(self, operation, arguments, operation_id):
        receipt = {
            "operation_id": operation_id,
            "operation": operation,
            "arguments": arguments,
            "authorization": "allowed",
            "status": "completed",
        }
        self.receipts.append(receipt)
        return {"status": "ok", "data": {"text": "example"}, "receipt": receipt}


broker = Broker()
notebook_result = broker.call("read", {"path": "parser.py"}, "notebook-cell-1:call-1")
prime_cell = {"status": "completed", "effects": "native_untracked"}
assert notebook_result["data"] == {"text": "example"}
assert {row["operation"] for row in broker.receipts} == {"read"}
assert len(broker.receipts) == 1
print({"brokered": broker.receipts, "prime": prime_cell})

## Different state contracts

Skein notebook PTC reconstructs only committed self-contained data cells. Prime restores bounded serializable values from a trusted dill snapshot. This difference is part of the ablation, not hidden by an adapter.


In [ ]:
safe_cells = ["target = 'parser.py'", "observations = {'branch': 'union'}"]
notebook_heap = {}
for cell in safe_cells:
    exec(cell, notebook_heap)
restored_heap = {}
for cell in safe_cells:
    exec(cell, restored_heap)
assert restored_heap["observations"] == notebook_heap["observations"]

prime_snapshot = {"target": "parser.py"}
prime_restored = dict(prime_snapshot)
assert prime_restored["target"] == "parser.py"
print({"notebook_replayed": True, "prime_snapshot_restored": True})

## Configuration

Use `notebook_ptc.enabled: true` and select `implementation: skein_notebook` or `prime_repl`. Prime also requires `prime_native_execution: true` and project trust. Keep model, task, budgets, and memory strategy fixed when comparing them.
